In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib scikit-learn tqdm
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q librosa soundfile tensorboard
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU不可用，请在运行时类型中选择T4 GPU')

# 训练技巧与实验方法论

## 学习目标

- 理解过拟合/欠拟合，学会用训练/验证集检测
- 掌握学习率、批大小、优化器的选择
- 掌握音频数据增强方法
- 学会用TensorBoard记录实验
- 这是"能跑通"到"能做研究"的关键一步


## 1. 过拟合与欠拟合

**过拟合**：模型在训练集上表现好，但在验证集上表现差——"死记硬背"。
**欠拟合**：模型在训练集上就表现不好——"没学够"。

我们来做一个实验：把训练集缩小到10个样本，观察过拟合。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchaudio.transforms as T
import torchaudio

plt.rcParams['font.size'] = 12

# 复用上节课的数据集和模型
from importlib.util import spec_from_file_location
# 直接重新定义（确保notebook自包含）

class SpeechNoiseDataset(torch.utils.data.Dataset):
    def __init__(self, n_samples=800, sr=16000, duration=0.5):
        self.sr = sr
        self.data = []
        self.labels = []
        n_sig = int(sr * duration)
        for i in range(n_samples):
            t = np.linspace(0, duration, n_sig, endpoint=False)
            if i < n_samples // 2:
                freq = np.random.uniform(100, 400)
                signal = sum(np.random.uniform(0.1, 0.4) * np.sin(2*np.pi*freq*k*t) for k in range(1, np.random.randint(3, 7)))
                signal += 0.02 * np.random.randn(n_sig)
                label = 0
            else:
                signal = np.random.uniform(0.05, 0.3) * np.random.randn(n_sig)
                label = 1
            self.data.append(torch.FloatTensor(signal))
            self.labels.append(label)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

class AudioCNN(nn.Module):
    def __init__(self, n_mels=64, n_classes=2):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=512, hop_length=160, n_mels=n_mels)
        self.db_transform = T.AmplitudeToDB(stype='power', top_db=80)
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, n_classes)
    def forward(self, x):
        mel = self.mel_transform(x)
        mel_db = self.db_transform(mel)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
        mel_db = mel_db.unsqueeze(1)
        f = self.features(mel_db).view(x.size(0), -1)
        return self.classifier(f)

print('数据和模型准备完成')

In [ ]:
# ====== 过拟合实验：缩小训练集 ======
np.random.seed(42)
torch.manual_seed(42)

full_dataset = SpeechNoiseDataset(n_samples=800)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset_full, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# 正常训练集 vs 极小训练集
train_dataset_small = Subset(train_dataset_full, range(10))  # 只有10个样本！

def train_model(train_ds, val_ds, epochs=30, lr=0.001):
    model = AudioCNN()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)
    
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    for epoch in range(epochs):
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for x, y in train_loader:
            out = model(x)
            loss = criterion(out, y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            t_loss += loss.item()
            t_correct += (out.argmax(1) == y).sum().item()
            t_total += y.size(0)
        train_losses.append(t_loss / len(train_loader))
        train_accs.append(t_correct / t_total)
        
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                out = model(x)
                v_loss += criterion(out, y).item()
                v_correct += (out.argmax(1) == y).sum().item()
                v_total += y.size(0)
        val_losses.append(v_loss / len(val_loader))
        val_accs.append(v_correct / v_total)
    
    return train_losses, val_losses, train_accs, val_accs

# 训练两个模型
print('训练正常数据量模型...')
tl_full, vl_full, ta_full, va_full = train_model(train_dataset_full, val_dataset)
print('训练极小数据量模型...')
tl_small, vl_small, ta_small, va_small = train_model(train_dataset_small, val_dataset)

In [ ]:
# 对比过拟合 vs 正常训练
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(tl_full, label='Train'); axes[0, 0].plot(vl_full, label='Val')
axes[0, 0].set_title('正常数据 - Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(ta_full, label='Train'); axes[0, 1].plot(va_full, label='Val')
axes[0, 1].set_title('正常数据 - Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(tl_small, label='Train'); axes[1, 0].plot(vl_small, label='Val')
axes[1, 0].set_title('极小数据(10样本) - Loss'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(ta_small, label='Train'); axes[1, 1].plot(va_small, label='Val')
axes[1, 1].set_title('极小数据(10样本) - Accuracy'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('\n观察：极小数据时，训练loss很低但验证loss很高 → 过拟合！')

### 对抗过拟合的方法

1. **更多数据**——最有效
2. **数据增强**——等价于增加数据
3. **Dropout**——随机丢弃神经元，防止过拟合
4. **权重衰减(L2正则)**——限制参数大小
5. **早停(Early Stopping)**——验证集性能不再提升时停止训练

## 2. 学习率、批大小、优化器

### 2.1 学习率的影响

学习率是最重要的超参数。我们来实验不同的学习率。

In [ ]:
# 学习率实验
lrs = [0.0001, 0.001, 0.01, 0.1]
results = {}

for lr in lrs:
    print(f'训练 lr={lr}...')
    tl, vl, ta, va = train_model(train_dataset_full, val_dataset, epochs=20, lr=lr)
    results[lr] = (tl, vl, ta, va)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for lr in lrs:
    tl, vl, ta, va = results[lr]
    axes[0].plot(tl, label=f'lr={lr}')
    axes[1].plot(va, label=f'lr={lr}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].set_title('Training Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy'); axes[1].set_title('Validation Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\n观察：lr太小→训练慢，lr太大→震荡或不收敛')

### 2.2 优化器选择

| 优化器 | 特点 | 推荐场景 |
|--------|------|----------|
| SGD | 简单，需要仔细调学习率 | 大规模训练，精调 |
| SGD+Momentum | 加速收敛 | 通用 |
| Adam | 自适应学习率，收敛快 | **默认选择**，研究初期 |
| AdamW | Adam + 权重衰减，更好泛化 | Transformer等大模型 |

**建议**：初期用Adam(lr=0.001)，后期如果需要精调，换成SGD+Momentum。

## 3. 音频数据增强

数据增强是提高模型泛化能力的重要手段，尤其在数据量有限时。

音频领域常用的增强方法：
1. **加噪**——模拟真实环境中的噪声
2. **时间拉伸**——改变语速
3. **频谱掩蔽(SpecAugment)**——遮住部分时间或频率区域

这些方法直接对接后续的DeepFilterNet模块——语音增强本质上就是在做"去噪"。

In [ ]:
# 音频数据增强演示
import torchaudio.transforms as T

# 生成一段模拟语音
sr = 16000
duration = 1.0
n_samples = int(sr * duration)
t = np.linspace(0, duration, n_samples, endpoint=False)
clean_signal = sum(0.3/k * np.sin(2*np.pi*200*k*t) for k in range(1, 5))
clean_tensor = torch.FloatTensor(clean_signal).unsqueeze(0)

# 1. 加噪
noise = 0.1 * torch.randn_like(clean_tensor)
noisy_signal = clean_tensor + noise

# 2. 频谱掩蔽 (SpecAugment)
mel_transform = T.MelSpectrogram(sample_rate=sr, n_fft=512, hop_length=160, n_mels=64)
db_transform = T.AmplitudeToDB(stype='power', top_db=80)
mel = mel_transform(clean_tensor)

freq_mask = T.FrequencyMasking(freq_mask_param=10)
time_mask = T.TimeMasking(time_mask_param=20)
mel_freq_masked = freq_mask(mel)
mel_time_masked = time_mask(mel)
mel_both_masked = time_mask(freq_mask(mel))

# 可视化
fig, axes = plt.subplots(1, 5, figsize=(20, 3))
titles = ['原始语谱图', '加噪', '频率掩蔽', '时间掩蔽', '频率+时间掩蔽']
mel_to_show = [
    db_transform(mel),
    db_transform(mel_transform(noisy_signal)),
    db_transform(mel_freq_masked),
    db_transform(mel_time_masked),
    db_transform(mel_both_masked),
]
for ax, m, title in zip(axes, mel_to_show, titles):
    ax.imshow(m[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(title)
    ax.set_ylabel('Mel Band') if ax == axes[0] else None
    ax.set_xlabel('Time Frame')
plt.tight_layout()
plt.show()

## 4. 实验管理：TensorBoard

在做研究时，每次实验都要记录——做了什么改动、效果如何。
**不做记录的实验等于没做。**

TensorBoard是PyTorch最常用的实验记录工具。

In [ ]:
# TensorBoard使用示例
from torch.utils.tensorboard import SummaryWriter

# 创建writer
writer = SummaryWriter('runs/experiment_1')

# 在训练循环中记录
model = AudioCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset_full, batch_size=32, shuffle=True)

# 训练5个epoch做演示
for epoch in range(5):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        out = model(batch_x)
        loss = criterion(out, batch_y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    
    # 记录到TensorBoard
    writer.add_scalar('Loss/train', total_loss / len(train_loader), epoch)
    writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)

writer.close()
print('TensorBoard日志已写入 runs/experiment_1')
print('在终端运行: tensorboard --logdir=runs')
print('然后在浏览器打开: http://localhost:6006')

### TensorBoard最佳实践

1. **每次实验用不同的文件夹**：`runs/lr_0.001`, `runs/lr_0.0001`
2. **记录所有超参数**：学习率、批大小、模型结构
3. **记录训练和验证的loss和accuracy**
4. **记录学习率变化**（如果用了scheduler）
5. **记录模型结构**：`writer.add_graph(model, input)`

## 本节要点

1. **过拟合**= 训练集好但验证集差 → 解决方法：更多数据、数据增强、Dropout、早停
2. **学习率**是最重要的超参数——太大震荡，太小慢
3. **Adam(lr=0.001)** 是研究初期的默认选择
4. **数据增强**对音频尤其重要：加噪、时间拉伸、频谱掩蔽
5. **实验必须记录**——TensorBoard是最常用的工具
6. **训练/验证/测试集划分**是做研究的基本规范


---
返回目录：[README.md](../README.md)